# Chapter 6 — Implementing MCP Client (Connecting Agent to MCP Server)

Now you’ll learn the most important integration step:

> How your agent connects to an MCP server and uses tools dynamically.

You already built the MCP server in Chapter 5.

Now we connect to it from an agent.

---

# 1️⃣ What MCP Client Does

MCP Client acts as bridge between:

```text
Agent (LLM + reasoning)
        ↓
MCP Client
        ↓
MCP Server (tools/resources/prompts)
```

Client responsibilities:

* Connect to server
* Discover tools
* Send tool calls
* Receive results
* Return results to agent

---

# 2️⃣ Internal Flow (Very Important)

Full lifecycle:

```text
Step 1: Client connects to MCP server

Step 2: Client asks → tools/list

Step 3: Server returns tool schemas

Step 4: Client gives tool schemas to LLM

Step 5: LLM selects tool

Step 6: Client sends tools/call

Step 7: Server executes tool

Step 8: Result returned to agent
```

This integrates directly into ReAct cycle.

---

# 3️⃣ Installing MCP Client SDK

Same SDK provides client:

```bash
pip install mcp
```

---

# 4️⃣ Connecting to MCP Server (Basic Example)

Assume your server is running locally.

### Client Code:

```python
import asyncio
from mcp.client.session import ClientSession
from mcp.client.stdio import stdio_client

async def main():
    async with stdio_client("python server.py") as (read, write):
        async with ClientSession(read, write) as session:

            # Discover tools
            tools = await session.list_tools()

            print("Available tools:")
            for tool in tools:
                print(tool.name)

asyncio.run(main())
```

This connects agent to MCP server.

---

# 5️⃣ Calling a Tool from MCP Client

Now let’s call tool.

```python
result = await session.call_tool(
    "get_weather",
    {"city": "Mumbai"}
)

print(result)
```

Server executes tool.

Returns result.

---

# 6️⃣ What Happens Internally

Behind the scenes:

Client sends:

```json
{
  "method": "tools/call",
  "params": {
    "name": "get_weather",
    "arguments": {
      "city": "Mumbai"
    }
  }
}
```

Server executes function.

Returns result.

Client forwards to agent.

---

# 7️⃣ Integrating with Your Agent (Important)

You already know agent structure:

```python
class Agent:
    tools = []
    llm = model
```

With MCP, tools become dynamic:

```python
tools = await session.list_tools()
```

Pass these schemas to LLM.

LLM decides tool.

Client executes tool.

---

# 8️⃣ Full Working Agent Example (Simplified)

```python
import asyncio
from mcp.client.session import ClientSession
from mcp.client.stdio import stdio_client

class MCPAgent:

    async def run(self):

        async with stdio_client("python server.py") as (read, write):

            async with ClientSession(read, write) as session:

                # discover tools
                tools = await session.list_tools()

                print("Discovered tools:", tools)

                # simulate LLM choosing tool
                result = await session.call_tool(
                    "get_weather",
                    {"city": "Delhi"}
                )

                print("Tool result:", result)

asyncio.run(MCPAgent().run())
```

This is minimal MCP-enabled agent.

---

# 9️⃣ How This Fits Into ReAct Cycle

Your ReAct loop becomes:

```text
User input
   ↓
LLM reasoning
   ↓
Select MCP tool
   ↓
MCP client executes tool
   ↓
Observation returned
   ↓
LLM continues reasoning
```

Exactly same ReAct pattern.

Only tool execution layer changes.

---

# 🔟 Resources Access Example

Reading resource:

```python
resource = await session.read_resource("customer://123")

print(resource)
```

No function call.

Direct resource access.

---

# 1️⃣1️⃣ Prompts Access Example

```python
prompt = await session.get_prompt(
    "review_code",
    {"code": "print('hello')"}
)
```

Returns reusable prompt template.

---

# 1️⃣2️⃣ Production Architecture Example

Real production agent:

```text
Agent
 ├── MCP Client
      ├── filesystem MCP server
      ├── database MCP server
      ├── github MCP server
      └── internal tools MCP server
```

Agent becomes extremely powerful.

---

# 1️⃣3️⃣ Key Insight (VERY IMPORTANT)

Your agent no longer owns tools.

It connects to tool ecosystem.

This makes agent:

* modular
* scalable
* extensible

---

# 1️⃣4️⃣ How ChatGPT / Claude / Cursor Work Internally

They connect to MCP servers like:

* filesystem
* git
* browser
* database

This is exactly how modern AI systems work.

You are now learning production-level architecture.

---

# Analogy — Installing Apps on Your Phone 📱

Your phone (Agent) does not contain all apps.

You install apps when needed.

Apps live outside core OS.

Same with MCP:

Agent = Phone
MCP Server = App Store
Tools = Apps
MCP Client = App installer

Agent installs and uses tools dynamically.

---

# Chapter 6 Summary

You learned:

* How MCP client connects to server
* How tools are discovered dynamically
* How tools are called
* How to integrate MCP with agent
* How ReAct cycle works with MCP

